# 07: Post-hoc checks

Added in September 2026, after notebook 06 had opened the lockbox. Nothing
here changes a sleeve, a parameter or the book, and none of it is a new
trial, so the registry is not touched. Three questions came up when the
backtests were audited:

1. The seasonality overlay halves the book on Mondays and restores it on
   Saturdays. `signals.seasonal_scale` scales the returns but never charges
   for those trades. What does the sleeve earn with every trade charged, and
   what does that do to the frozen book, lockbox included?
2. Orderflow and carry pay the limit-order rate (7 bps) on every rebalance.
   How much is left at 14 and 20 bps? (dev and gate only)
3. How much of the carry sleeve's P&L comes from funding and how much from
   price? (dev and gate only)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from quantlib import backtest, metrics, robustness, signals, strategies

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"

DEV = slice("2020-01-01", "2024-07-31")
GATE = slice("2024-08-01", "2025-06-30")
LOCKBOX = slice("2025-07-01", "2026-07-06")
FULL = slice("2020-01-01", "2026-07-06")
WINDOWS = {"dev": DEV, "gate": GATE, "lockbox": LOCKBOX, "full": FULL}

price = pd.read_parquet(PROC / "price.parquet")
returns = pd.read_parquet(PROC / "returns.parquet")
universe = pd.read_parquet(PROC / "universe.parquet")
taker = pd.read_parquet(PROC / "taker_imbalance.parquet")
funding = pd.read_parquet(PROC / "funding.parquet").reindex(columns=universe.columns)
bench = pd.read_parquet(PROC / "benchmarks.parquet")
asrun = pd.read_parquet(PROC / "combined_portfolio.parquet")

## 1. Seasonality with every trade charged

First rebuild the as-run book from the frozen sleeves and check it matches
`combined_portfolio.parquet` exactly, so that the only difference in the
re-score below is the extra cost.

In [2]:
seas = strategies.seasonal_momentum_sleeve(price, returns, universe)
orderflow = strategies.orderflow_sleeve(taker, returns, universe)
carry = strategies.carry_sleeve(funding, returns, universe)


def combine(seasonality):
    """Walk-forward and equal-weight books with notebook 06's settings."""
    sleeves = pd.concat([seasonality, orderflow, carry], axis=1, sort=True).dropna()
    wf, _ = robustness.walk_forward_weights(sleeves, train_days=756, step_days=63, min_train=252)
    return wf, sleeves.mean(axis=1)


wf_asrun, ew_asrun = combine(seas)
for name, mine, saved in [("walk_forward", wf_asrun, asrun["walk_forward"]),
                          ("equal_weight", ew_asrun, asrun["equal_weight"])]:
    both = pd.concat([mine, saved], axis=1, sort=True).dropna()
    same = np.array_equal(both.iloc[:, 0].to_numpy(), both.iloc[:, 1].to_numpy())
    print(f"{name}: {len(both)} days, identical to notebook 06: {same}")
    assert same

walk_forward: 1736 days, identical to notebook 06: True
equal_weight: 2492 days, identical to notebook 06: True


Now the charged versions. `charge_resizing=True` rebuilds the positions
actually held each day (momentum weights x vol scale x day multiplier) and
pays 20 bps on every change. The untilted sleeve is charged the same way,
because notebook 02's rule compared the overlay with it.

In [3]:
seas_charged = strategies.seasonal_momentum_sleeve(price, returns, universe, charge_resizing=True)
untilted = strategies.seasonal_momentum_sleeve(price, returns, universe, weekend=1.0, weekday=1.0)
untilted_charged = strategies.seasonal_momentum_sleeve(price, returns, universe, weekend=1.0,
                                                       weekday=1.0, charge_resizing=True)


def sharpe_row(s, windows):
    return {w: metrics.sharpe(s.loc[WINDOWS[w]].dropna()) for w in windows}


sleeve_table = pd.DataFrame({
    "seasonality, as run": sharpe_row(seas, ["dev", "gate", "lockbox"]),
    "seasonality, all trades charged": sharpe_row(seas_charged, ["dev", "gate", "lockbox"]),
    "untilted momentum, as run": sharpe_row(untilted, ["dev", "gate"]),
    "untilted momentum, all trades charged": sharpe_row(untilted_charged, ["dev", "gate"]),
}).T
sleeve_table.round(3)

,dev,gate,lockbox
"seasonality, as run",1.790,1.243,-0.565
"seasonality, all trades charged",1.515,0.993,-0.892
"untilted momentum, as run",1.650,0.434,NaN
"untilted momentum, all trades charged",1.636,0.421,NaN


The overlay's extra trading, as a yearly cost on dev:

In [4]:
extra = (seas.loc[DEV] - seas_charged.loc[DEV]).mean() * 365
print(f"extra cost of resizing on dev: {extra:.2%} a year")

extra cost of resizing on dev: 2.69% a year


The same frozen book with the charged seasonality sleeve. Weights,
windows and the walk-forward settings are unchanged.

In [5]:
wf_charged, ew_charged = combine(seas_charged)


def book_stats(r):
    ab = metrics.alpha_beta(r, bench["BTC"], bench["MKT"], names=["BTC", "MKT"])
    return {
        "sharpe": metrics.sharpe(r),
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "max_dd": metrics.max_drawdown(r),
        "beta_btc": ab["beta_BTC"],
        "alpha_ann": ab["alpha_ann"],
        "alpha_t": ab["alpha_tstat"],
    }


rows = {}
for label, wf, ew in [("as run", wf_asrun, ew_asrun), ("seasonality charged", wf_charged, ew_charged)]:
    for w in ["dev", "gate", "lockbox", "full"]:
        rows[(label, "walk-forward", w)] = book_stats(wf.loc[WINDOWS[w]].dropna())
        rows[(label, "equal weight", w)] = book_stats(ew.loc[WINDOWS[w]].dropna())
book_table = pd.DataFrame(rows).T
book_table.round(3)

sharpe  ann_return  ann_vol  max_dd  \
as run              walk-forward dev       0.397       0.029    0.081  -0.081   
                    equal weight dev       1.681       0.250    0.138  -0.101   
                    walk-forward gate      2.519       0.216    0.079  -0.039   
                    equal weight gate      2.903       0.400    0.118  -0.042   
                    walk-forward lockbox   1.458       0.114    0.076  -0.047   
                    equal weight lockbox   1.233       0.162    0.129  -0.092   
                    walk-forward full      1.020       0.081    0.079  -0.081   
                    equal weight full      1.763       0.256    0.134  -0.101   
seasonality charged walk-forward dev      -0.102      -0.012    0.085  -0.130   
                    equal weight dev       1.616       0.239    0.138  -0.111   
                    walk-forward gate      2.514       0.214    0.078  -0.040   
                    equal weight gate      2.833       0.388    0.118  -0.042   
                    walk-forward lockbox   1.490       0.125    0.081  -0.052   
                    equal weight lockbox   1.158       0.151    0.129  -0.096   
                    walk-forward full      0.703       0.057    0.083  -0.132   
                    equal weight full      1.696       0.244    0.134  -0.111   

                                          beta_btc  alpha_ann  alpha_t  
as run              walk-forward dev         0.003      0.036    0.804  
                    equal weight dev        -0.016      0.224    3.457  
                    walk-forward gate       -0.046      0.230    2.752  
                    equal weight gate       -0.043      0.372    3.348  
                    walk-forward lockbox    -0.020      0.095    1.253  
                    equal weight lockbox    -0.085      0.134    0.994  
                    walk-forward full       -0.013      0.083    2.325  
                    equal weight full       -0.022      0.236    4.418  
seasonality charged walk-forward dev         0.003     -0.004   -0.094  
                    equal weight dev        -0.016      0.215    3.319  
                    walk-forward gate       -0.039      0.223    2.715  
                    equal weight gate       -0.043      0.364    3.273  
                    walk-forward lockbox    -0.016      0.108    1.331  
                    equal weight lockbox    -0.085      0.124    0.922  
                    walk-forward full       -0.012      0.060    1.627  
                    equal weight full       -0.022      0.227    4.250

Significance of the re-scored walk-forward book, with the same 48-trial
count and bootstrap settings as notebook 06.

In [6]:
sig_rows = {}
for label, wf in [("as run", wf_asrun), ("seasonality charged", wf_charged)]:
    lk = wf.loc[LOCKBOX].dropna()
    full = wf.loc[FULL].dropna()
    lo, hi = metrics.bootstrap_sharpe_ci(lk, n_boot=2000, method="block", seed=42)
    sig_rows[label] = {
        "lockbox sharpe": metrics.sharpe(lk),
        "lockbox 95% CI low": lo,
        "lockbox 95% CI high": hi,
        "deflated sharpe, full": metrics.deflated_sharpe(full, n_trials=48),
        "deflated sharpe, lockbox": metrics.deflated_sharpe(lk, n_trials=48),
    }
pd.DataFrame(sig_rows).T.round(3)

,lockbox sharpe,lockbox 95% CI low,lockbox 95% CI high,"deflated sharpe, full","deflated sharpe, lockbox"
as run,1.458,-0.423,3.444,0.490,0.205
seasonality charged,1.490,-0.397,3.459,0.234,0.214


## 2. Orderflow and carry at higher costs (dev and gate only)

In [7]:
upto_gate = slice(None, "2025-06-30")
stress = {}
for bps in [7, 14, 20]:
    of = strategies.orderflow_sleeve(taker.loc[upto_gate], returns.loc[upto_gate],
                                     universe.loc[upto_gate], cost_bps=bps)
    ca = strategies.carry_sleeve(funding.loc[upto_gate], returns.loc[upto_gate],
                                 universe.loc[upto_gate], cost_bps=bps)
    for name, s in [("orderflow", of), ("carry", ca)]:
        stress[(name, bps)] = sharpe_row(s.loc["2020-01-01":], ["dev", "gate"])
cost_table = pd.DataFrame(stress).T
cost_table.index.names = ["sleeve", "cost_bps"]
cost_table.round(3)

,,dev,gate
sleeve,cost_bps,,
orderflow,7,0.886,2.118
carry,7,1.007,2.186
orderflow,14,0.611,1.378
carry,14,0.840,2.032
orderflow,20,0.374,0.745
carry,20,0.696,1.900


## 3. Carry: funding leg vs price leg (dev and gate only)

In [8]:
fuv = universe & funding.notna()
w = signals.signal_to_weights(-signals.cross_sectional_zscore(funding.rolling(7).mean(), fuv),
                              fuv, long_short=True)
price_leg = backtest.run(w, returns, cost_bps=7).net_returns
funding_leg = -(w.shift(1) * funding).sum(axis=1)
split = {}
for w_name in ["dev", "gate"]:
    sl = WINDOWS[w_name]
    p, f = price_leg.loc[sl], funding_leg.loc[sl]
    split[w_name] = {
        "price leg, mean %/yr": p.mean() * 365,
        "funding leg, mean %/yr": f.mean() * 365,
        "funding share of total": f.mean() / (p.mean() + f.mean()),
        "price leg sharpe": metrics.sharpe(p),
        "funding leg sharpe": metrics.sharpe(f),
    }
carry_table = pd.DataFrame(split).T
carry_table.round(3)

,"price leg, mean %/yr","funding leg, mean %/yr",funding share of total,price leg sharpe,funding leg sharpe
dev,0.114,0.193,0.630,0.379,4.786
gate,0.412,0.269,0.394,1.318,12.340


## Save for the report

In [9]:
pd.concat([seas.rename("seasonality"), orderflow.rename("orderflow"), carry.rename("carry")],
          axis=1).to_parquet(PROC / "sleeves_full.parquet")
pd.DataFrame({
    "seasonality_charged": seas_charged,
    "walk_forward_charged": wf_charged,
    "equal_weight_charged": ew_charged,
}).to_parquet(PROC / "posthoc_book.parquet")
cost_table.reset_index().to_csv(PROC / "posthoc_cost_stress.csv", index=False)
carry_table.rename_axis("window").reset_index().to_csv(PROC / "posthoc_carry_split.csv", index=False)
print("saved sleeves_full, posthoc_book, posthoc_cost_stress, posthoc_carry_split")

saved sleeves_full, posthoc_book, posthoc_cost_stress, posthoc_carry_split


## Conclusion

**Seasonality.** Charging every resizing trade costs the sleeve about 2.7%
a year on dev. Its Sharpe falls from 1.79 to 1.52 on dev, from 1.24 to 0.99
on the gate and from -0.56 to -0.89 on the lockbox. The untilted momentum
sleeve, charged the same way, scores 1.64 on dev and 0.42 on the gate. The
overlay still wins on the gate but loses on dev, so under notebook 02's
rule (beat the untilted sleeve on both windows) it would not have been
selected. The error was in the cost accounting; the signal itself is fine.

**The book.** On the lockbox the effect is small and goes both ways:
walk-forward moves from 1.458 to 1.490 and equal weight from 1.233 to
1.158. Walk-forward gains because a weaker seasonality sleeve gets less
weight in the training windows, and seasonality lost money on the lockbox.
Over the full sample the walk-forward Sharpe falls from 1.02 to 0.70 and
its deflated Sharpe from 0.49 to 0.23. On the lockbox the deflated Sharpe
is 0.21, against 0.20 as run, and beta to BTC stays close to zero. The
conclusion of notebook 06 stands: a positive, low-beta result that is not
statistically proven.

**Costs.** At 20 bps instead of 7, orderflow falls to 0.37 on dev and 0.75
on the gate, and carry to 0.70 and 1.90. Both stay positive, but orderflow
depends heavily on getting limit-order fills. Carry is less sensitive: it
turns over about a fifth of its book a day, and most of its dev P&L is
funding.

**Carry P&L.** On dev, funding brings in 19.3% a year out of the sleeve's
30.7% (arithmetic means), 63% of the total. On the gate the price leg did
more, 41.2% against 26.9%. The funding leg on its own has a very high
Sharpe (4.8 on dev) because the payments are small and steady; the risk
sits in the price leg.